# 📊 Data Analysis — 10-Week Course
## Week 4: Exploratory Data Analysis (EDA)

---
### Learning Objectives
By the end of this week, you will be able to:
- Compute and interpret descriptive statistics (mean, median, variance, skewness, kurtosis)
- Create effective visual summaries: histograms, boxplots, scatterplots, pair plots
- Identify patterns, trends, anomalies, and relationships in data
- Write a structured EDA narrative

### Outline
1. What Is EDA?
2. Descriptive Statistics Deep Dive
3. Univariate Analysis
4. Bivariate Analysis
5. Multivariate Analysis
6. Writing an EDA Narrative
7. Lab Exercises
8. Assignment

---
## 1. What Is EDA?

**Exploratory Data Analysis (EDA)**, introduced by John Tukey (1977), is an approach to analysing data sets to summarise their main characteristics — often using visual methods — *before* applying formal statistical models.

### EDA Goals
| Goal | Question |
|---|---|
| **Understand distributions** | What shape does each variable take? |
| **Detect anomalies** | Are there outliers or impossible values? |
| **Find relationships** | Do variables move together? |
| **Generate hypotheses** | What patterns deserve formal testing? |
| **Guide modelling** | What transformations or features are needed? |

### EDA vs Confirmatory Analysis
| | EDA | Confirmatory |
|---|---|---|
| Purpose | Discover | Verify |
| Tools | Plots, summaries | Hypothesis tests, CIs |
| Mindset | Open-ended | Pre-specified |
| Risk | Data dredging | Type I error |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

# ── Load / recreate the cleaned Africa Health dataset ──────────────────
import os
DATA_FILE = "africa_health_cleaned.csv"

if os.path.exists(DATA_FILE):
    df = pd.read_csv(DATA_FILE)
    print(f"Loaded {DATA_FILE}: {df.shape}")
else:
    # Recreate the clean dataset from scratch
    np.random.seed(42)
    N = 54
    countries = [
        "Nigeria","Ethiopia","Egypt","DR Congo","Tanzania","South Africa","Kenya",
        "Uganda","Algeria","Sudan","Morocco","Mozambique","Ghana","Angola","Cameroon",
        "Madagascar","Côte d'Ivoire","Niger","Burkina Faso","Mali","Malawi","Zambia",
        "Senegal","Chad","Somalia","Zimbabwe","Guinea","Rwanda","Benin","Burundi",
        "Tunisia","South Sudan","Togo","Sierra Leone","Libya","Congo","Liberia",
        "Mauritania","Eritrea","Namibia","Gambia","Botswana","Gabon","Lesotho",
        "Guinea-Bissau","Equatorial Guinea","Mauritius","Eswatini","Djibouti",
        "Comoros","Cabo Verde","Sao Tome","Seychelles","São Tomé"
    ]
    regions = (["West Africa"]*16+["East Africa"]*14+
               ["North Africa"]*6+["Central Africa"]*8+["Southern Africa"]*10)
    df = pd.DataFrame({
        "country"           : countries,
        "region"            : regions,
        "life_expectancy"   : np.clip(np.random.normal(63, 8, N), 45, 80),
        "infant_mortality"  : np.clip(np.random.exponential(35, N), 5, 120),
        "maternal_mortality": np.clip(np.random.exponential(350, N), 20, 2000),
        "hiv_prevalence"    : np.clip(np.random.exponential(3.5, N), 0.1, 28),
        "health_expenditure": np.clip(np.random.normal(5.2, 2.1, N), 1, 12),
        "gdp_per_capita"    : np.clip(np.exp(np.random.normal(7.2, 1.2, N)), 300, 15000),
        "vaccination_dtp3"  : np.clip(np.random.normal(78, 18, N), 20, 99),
        "water_access"      : np.clip(np.random.normal(68, 22, N), 15, 99),
    })
    print(f"Recreated dataset: {df.shape}")

numeric_cols = df.select_dtypes(include="number").columns.tolist()
print("Numeric columns:", numeric_cols)
df.head()

---
## 2. Descriptive Statistics Deep Dive

### Measures of Central Tendency
| Measure | Formula | Best When |
|---|---|---|
| **Mean** | Σx / n | Symmetric, no outliers |
| **Median** | Middle value | Skewed data, outliers present |
| **Mode** | Most frequent | Categorical data |

### Measures of Spread
| Measure | Formula | Interpretation |
|---|---|---|
| **Range** | max − min | Sensitive to extremes |
| **IQR** | Q3 − Q1 | Robust spread (middle 50%) |
| **Variance** | Σ(x−μ)² / n | Average squared deviation |
| **Std Dev** | √variance | Same units as data |
| **CV** | σ/μ × 100% | Relative spread (unit-free) |

### Shape Statistics
| Statistic | Interpretation |
|---|---|
| **Skewness > 0** | Right-skewed (tail on right) |
| **Skewness < 0** | Left-skewed (tail on left) |
| **Kurtosis > 3** | Leptokurtic (heavy tails) |
| **Kurtosis < 3** | Platykurtic (light tails) |

In [ ]:
def full_summary(df, cols):
    """Extended descriptive statistics beyond pandas .describe()"""
    rows = []
    for col in cols:
        s = df[col].dropna()
        rows.append({
            "Variable"  : col,
            "Count"     : len(s),
            "Mean"      : round(s.mean(), 2),
            "Median"    : round(s.median(), 2),
            "Std Dev"   : round(s.std(), 2),
            "IQR"       : round(s.quantile(0.75) - s.quantile(0.25), 2),
            "CV %"      : round(s.std() / s.mean() * 100, 1),
            "Skewness"  : round(s.skew(), 3),
            "Kurtosis"  : round(s.kurt() + 3, 3),   # convert to Pearson kurtosis
            "Min"       : round(s.min(), 2),
            "Max"       : round(s.max(), 2),
        })
    return pd.DataFrame(rows).set_index("Variable")

summary = full_summary(df, numeric_cols)
print("=== Extended Descriptive Statistics ===")
print(summary.to_string())

In [ ]:
# Interpret skewness automatically
print("=== Skewness Interpretation ===")
for col in numeric_cols:
    sk = df[col].skew()
    if   sk >  1:   label = "strongly right-skewed ↗"
    elif sk >  0.5: label = "moderately right-skewed →"
    elif sk < -1:   label = "strongly left-skewed  ↙"
    elif sk < -0.5: label = "moderately left-skewed ←"
    else:           label = "approximately symmetric ↔"
    print(f"  {col:<22} skew={sk:+.3f}  {label}")

---
## 3. Univariate Analysis

Univariate analysis examines **one variable at a time**:
- **Histograms** — distribution shape, modality, gaps
- **Box plots** — centre, spread, outliers at a glance
- **Density plots** — smooth distribution estimate
- **Bar charts** — frequency of categorical values

In [ ]:
# Histogram grid for all numeric variables
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

colors = plt.cm.tab10.colors

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    data = df[col].dropna()

    ax.hist(data, bins=15, color=colors[i], alpha=0.75, edgecolor="white")

    # Add KDE overlay
    kde_x = np.linspace(data.min(), data.max(), 200)
    kde   = stats.gaussian_kde(data)
    ax2   = ax.twinx()
    ax2.plot(kde_x, kde(kde_x), color="black", lw=1.5, alpha=0.7)
    ax2.set_yticks([])
    ax2.spines["right"].set_visible(False)
    ax2.spines["top"].set_visible(False)

    # Mean and median lines
    ax.axvline(data.mean(),   color="red",   ls="--", lw=1.2, label="Mean")
    ax.axvline(data.median(), color="green", ls="--", lw=1.2, label="Median")

    ax.set_title(col.replace("_", " ").title(), fontweight="bold", fontsize=9)
    ax.set_xlabel("")
    if i == 0:
        ax.legend(fontsize=7, loc="upper right")

# Hide the empty 8th panel
axes[-1].set_visible(False)

fig.suptitle("Univariate Distributions — Africa Health Dataset",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("week4_histograms.png", bbox_inches="tight")
plt.show()
print("Key observations:")
print("  • life_expectancy   → roughly normal, slight left skew")
print("  • infant_mortality  → right-skewed (exponential shape)")
print("  • maternal_mortality → strongly right-skewed")
print("  • hiv_prevalence    → right-skewed, many low values")
print("  • gdp_per_capita    → strongly right-skewed (log-normal)")

In [ ]:
# Box plots — outlier detection
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    bp = ax.boxplot(
        df[col].dropna(),
        vert=True,
        patch_artist=True,
        boxprops=dict(facecolor=colors[i], alpha=0.6),
        medianprops=dict(color="black", lw=2),
        flierprops=dict(marker="o", markerfacecolor="red", markersize=5),
    )
    # Count outliers
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()

    ax.set_title(f"{col.replace('_',' ').title()}\n(outliers: {n_out})",
                 fontsize=8, fontweight="bold")
    ax.set_xticks([])

axes[-1].set_visible(False)
fig.suptitle("Box Plots — Outlier Overview", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("week4_boxplots.png", bbox_inches="tight")
plt.show()

In [ ]:
# Categorical variable: region frequency + mean life expectancy
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Country count per region
region_counts = df["region"].value_counts()
axes[0].bar(region_counts.index, region_counts.values,
            color="#3498DB", edgecolor="white", alpha=0.85)
axes[0].set_title("Countries per Region", fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=15)

# Mean life expectancy per region with error bars (std)
grp = df.groupby("region")["life_expectancy"].agg(["mean", "std"]).sort_values("mean")
axes[1].barh(grp.index, grp["mean"], xerr=grp["std"],
             color="#E74C3C", edgecolor="white", alpha=0.85, capsize=4)
axes[1].set_title("Mean Life Expectancy by Region (±1 SD)", fontweight="bold")
axes[1].set_xlabel("Years")

plt.tight_layout()
plt.savefig("week4_regional_bars.png", bbox_inches="tight")
plt.show()

---
## 4. Bivariate Analysis

Bivariate analysis examines the **relationship between two variables**.

| Variable Types | Tool |
|---|---|
| Numeric vs Numeric | Scatter plot, correlation |
| Categorical vs Numeric | Box/violin plot, grouped bar |
| Categorical vs Categorical | Contingency table, stacked bar |

### Pearson vs Spearman Correlation
| | Pearson | Spearman |
|---|---|---|
| Measures | Linear relationship | Monotonic relationship |
| Assumes | Normal distribution | No distribution assumption |
| Sensitive to outliers | Yes | No |
| Range | −1 to +1 | −1 to +1 |

In [ ]:
# Scatter: GDP vs Life Expectancy  — coloured by region
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = {"West Africa": "#3498DB", "East Africa": "#2ECC71",
           "North Africa": "#E74C3C", "Central Africa": "#9B59B6",
           "Southern Africa": "#F39C12"}

for region, grp in df.groupby("region"):
    axes[0].scatter(grp["gdp_per_capita"], grp["life_expectancy"],
                    label=region, color=palette[region],
                    alpha=0.8, edgecolors="white", s=70)

# Fit a linear trend on log(GDP)
log_gdp = np.log(df["gdp_per_capita"])
m, b = np.polyfit(log_gdp, df["life_expectancy"], 1)
x_fit = np.linspace(df["gdp_per_capita"].min(), df["gdp_per_capita"].max(), 200)
axes[0].plot(x_fit, m * np.log(x_fit) + b, "k--", lw=1.5, label="Log-linear fit")

axes[0].set_xlabel("GDP per Capita (USD)")
axes[0].set_ylabel("Life Expectancy (years)")
axes[0].set_title("GDP vs Life Expectancy", fontweight="bold")
axes[0].legend(fontsize=7, loc="lower right")

# Same plot with log x-axis
for region, grp in df.groupby("region"):
    axes[1].scatter(grp["gdp_per_capita"], grp["life_expectancy"],
                    color=palette[region], alpha=0.8, edgecolors="white", s=70)
axes[1].set_xscale("log")
axes[1].set_xlabel("GDP per Capita (USD, log scale)")
axes[1].set_ylabel("Life Expectancy (years)")
axes[1].set_title("GDP vs Life Expectancy (log scale)", fontweight="bold")

# Pearson and Spearman correlations
r_p, p_p = stats.pearsonr(log_gdp, df["life_expectancy"])
r_s, p_s = stats.spearmanr(df["gdp_per_capita"], df["life_expectancy"])
axes[1].text(0.05, 0.95,
             f"Pearson (log GDP): r={r_p:.3f}, p={p_p:.4f}\n"
             f"Spearman:         ρ={r_s:.3f}, p={p_s:.4f}",
             transform=axes[1].transAxes, va="top", fontsize=8,
             bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

plt.suptitle("Week 4 — Bivariate: GDP vs Life Expectancy", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("week4_scatter_gdp_le.png", bbox_inches="tight")
plt.show()

In [ ]:
# Correlation matrix heatmap
corr = df[numeric_cols].corr(method="pearson")
spearman_corr = df[numeric_cols].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, matrix, title in zip(
    axes,
    [corr, spearman_corr],
    ["Pearson Correlation", "Spearman Correlation"]
):
    mask = np.triu(np.ones_like(matrix, dtype=bool), k=1)
    sns.heatmap(
        matrix, mask=mask, annot=True, fmt=".2f",
        cmap="RdYlGn", center=0, vmin=-1, vmax=1,
        square=True, linewidths=0.5, ax=ax,
        cbar_kws={"shrink": 0.8}
    )
    ax.set_title(title, fontweight="bold", pad=10)
    ax.tick_params(axis="x", rotation=45)

plt.suptitle("Correlation Matrices — Africa Health Dataset",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("week4_correlation.png", bbox_inches="tight")
plt.show()

# Top correlations with life expectancy
print("Top correlations with life_expectancy:")
le_corr = corr["life_expectancy"].drop("life_expectancy").sort_values()
for var, r in le_corr.items():
    direction = "↑" if r > 0 else "↓"
    print(f"  {var:<25} r = {r:+.3f} {direction}")

In [ ]:
# Categorical vs Numeric: region vs key health indicators
indicators = ["life_expectancy", "infant_mortality", "vaccination_dtp3", "water_access"]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

region_order = df.groupby("region")["life_expectancy"].median().sort_values().index.tolist()
pal = [palette[r] for r in region_order]

for i, col in enumerate(indicators):
    ax = axes[i]
    sns.violinplot(
        data=df, x="region", y=col,
        order=region_order, palette=pal,
        inner="box", alpha=0.8, ax=ax
    )
    sns.stripplot(
        data=df, x="region", y=col,
        order=region_order, color="black",
        size=3, alpha=0.5, jitter=True, ax=ax
    )
    ax.set_title(col.replace("_", " ").title(), fontweight="bold")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)

fig.suptitle("Regional Distributions — Violin + Strip Plots",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("week4_violin_region.png", bbox_inches="tight")
plt.show()

---
## 5. Multivariate Analysis

Multivariate analysis examines **three or more variables simultaneously**, revealing interactions that univariate/bivariate analyses miss.

### Key Techniques
| Technique | Purpose |
|---|---|
| **Pair plot** | All pairwise relationships in one figure |
| **Bubble chart** | 3 quantitative + 1 categorical |
| **Heatmap** | Matrix view of correlations or aggregates |
| **Facet grid** | Same plot for each category |
| **3D scatter** | Three numeric variables (use sparingly) |

In [ ]:
# Pair plot (subset of key variables)
pair_vars = ["life_expectancy", "infant_mortality",
             "gdp_per_capita", "vaccination_dtp3", "water_access"]

pg = sns.pairplot(
    df[pair_vars + ["region"]],
    hue="region",
    palette=palette,
    diag_kind="kde",
    plot_kws={"alpha": 0.6, "s": 40},
    diag_kws={"fill": True, "alpha": 0.4},
)
pg.figure.suptitle("Pair Plot — Key Health & Economic Variables",
                   y=1.02, fontsize=13, fontweight="bold")
pg.figure.savefig("week4_pairplot.png", bbox_inches="tight")
plt.show()
print("Observations from the pair plot:")
print("  1. GDP and life expectancy show a positive log-linear relationship")
print("  2. Infant mortality and life expectancy are negatively correlated")
print("  3. Water access and vaccination cluster together")
print("  4. North Africa countries tend to occupy the high-LE / high-GDP corner")

In [ ]:
# Bubble chart: GDP vs LE, bubble size = vaccination rate, colour = region
fig, ax = plt.subplots(figsize=(12, 7))

for region, grp in df.groupby("region"):
    sizes = (grp["vaccination_dtp3"] / grp["vaccination_dtp3"].max()) * 600 + 50
    sc = ax.scatter(
        np.log(grp["gdp_per_capita"]),
        grp["life_expectancy"],
        s=sizes,
        alpha=0.7,
        color=palette[region],
        edgecolors="white",
        linewidths=0.8,
        label=region,
    )

# Annotate a few notable countries
for _, row in df[df["country"].isin(["Nigeria","South Africa","Rwanda","Egypt","Mauritius"])].iterrows():
    ax.annotate(
        row["country"],
        (np.log(row["gdp_per_capita"]), row["life_expectancy"]),
        textcoords="offset points", xytext=(6, 4), fontsize=8
    )

ax.set_xlabel("Log GDP per Capita", fontsize=11)
ax.set_ylabel("Life Expectancy (years)", fontsize=11)
ax.set_title("Bubble Chart: GDP vs Life Expectancy\n(bubble size = DTP3 vaccination rate)",
             fontweight="bold")
ax.legend(title="Region", fontsize=9)

# Size legend
for pct, label in [(20,"20%"),(60,"60%"),(99,"99%")]:
    ax.scatter([], [], s=(pct/99)*600+50, c="grey", alpha=0.5, label=f"Vax: {label}")
ax.legend(title="Region / Vaccination", fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig("week4_bubble.png", bbox_inches="tight")
plt.show()

In [ ]:
# FacetGrid: Water access vs Life Expectancy per region
g = sns.FacetGrid(
    df, col="region", col_wrap=3,
    height=3.5, aspect=1.1,
    hue="region", palette=palette
)
g.map_dataframe(sns.regplot,
                x="water_access", y="life_expectancy",
                scatter_kws={"alpha": 0.7, "s": 50},
                line_kws={"lw": 2})
g.set_axis_labels("Water Access (%)", "Life Expectancy (yrs)")
g.set_titles(col_template="{col_name}")
g.figure.suptitle("Water Access vs Life Expectancy by Region",
                  fontsize=13, fontweight="bold", y=1.02)
g.figure.savefig("week4_facetgrid.png", bbox_inches="tight")
plt.show()

---
## 6. Writing an EDA Narrative

A good EDA report answers these questions in order:

1. **Overview** — How many rows/columns? What are the types and ranges?
2. **Data quality** — Missing values, outliers, inconsistencies
3. **Univariate findings** — What are the distributions of key variables?
4. **Bivariate findings** — What are the strongest relationships?
5. **Multivariate patterns** — How do groups differ? What interactions exist?
6. **Hypotheses** — What should be tested formally in Week 6?

### Structure Template
```
DATASET: <name>, <rows> rows × <cols> columns
PERIOD / SCOPE: <year/geography>

DATA QUALITY
  Missing: <summary>
  Outliers: <summary>

KEY FINDINGS
  1. <univariate observation>
  2. <bivariate observation>
  3. <regional/group pattern>

HYPOTHESES FOR TESTING
  H1: ...
  H2: ...
```

In [ ]:
# Auto-generate a structured EDA narrative
def eda_narrative(df, target="life_expectancy"):
    num_cols = df.select_dtypes(include="number").columns.tolist()
    cat_cols = df.select_dtypes(include="object").columns.tolist()

    missing_total = df.isnull().sum().sum()
    target_data   = df[target]

    # Outlier count per column
    outlier_counts = {}
    for col in num_cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        n = ((df[col] < Q1-1.5*IQR) | (df[col] > Q3+1.5*IQR)).sum()
        if n > 0:
            outlier_counts[col] = n

    # Top correlations
    corr = df[num_cols].corr()[target].drop(target).sort_values()

    print("=" * 60)
    print("EDA NARRATIVE REPORT")
    print("=" * 60)
    print(f"\nDATASET: Africa Health, {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"SCOPE: 54 African countries, 2022")
    print(f"\nDATA QUALITY")
    print(f"  Total missing values : {missing_total}")
    print(f"  Columns with outliers: {list(outlier_counts.keys())}")
    print(f"  Outlier counts       : {outlier_counts}")

    print(f"\nKEY UNIVARIATE FINDINGS")
    print(f"  {target}: mean={target_data.mean():.1f}, "
          f"sd={target_data.std():.1f}, skew={target_data.skew():.2f}")
    for col in num_cols:
        sk = df[col].skew()
        if abs(sk) > 1:
            print(f"  {col}: strongly {'right' if sk>0 else 'left'}-skewed (skew={sk:.2f})")

    print(f"\nSTRONGEST CORRELATIONS WITH {target.upper()}")
    for var, r in list(corr.items())[:3] + list(corr.items())[-3:]:
        print(f"  {var:<25} r = {r:+.3f}")

    print(f"\nREGIONAL PATTERNS")
    for region, grp in df.groupby("region"):
        print(f"  {region:<20} mean LE = {grp[target].mean():.1f} yrs")

    print(f"\nHYPOTHESES FOR FORMAL TESTING")
    print("  H1: North African countries have significantly higher life expectancy")
    print("      than Sub-Saharan African countries.")
    print("  H2: GDP per capita is a significant positive predictor of life expectancy.")
    print("  H3: Countries with DTP3 > 80% have lower infant mortality.")
    print("=" * 60)

eda_narrative(df)

---
## 7. Lab Exercises

### Lab 1: Extended Summary Statistics
Compute the 10th and 90th percentiles of `maternal_mortality` and `hiv_prevalence`.  
Interpret what the values mean in context.

In [ ]:
# Lab 1
for col in ["maternal_mortality", "hiv_prevalence"]:
    p10 = df[col].quantile(0.10)
    p90 = df[col].quantile(0.90)
    # TODO: Print the 10th and 90th percentile with a clear label
    # TODO: Write a one-sentence interpretation

### Lab 2: Relationship Exploration
Create a 1×3 scatter plot figure showing:  
- `health_expenditure` vs `life_expectancy`
- `vaccination_dtp3` vs `infant_mortality`
- `water_access` vs `maternal_mortality`  

Add the Pearson r value to each panel.

In [ ]:
# Lab 2
pairs = [
    ("health_expenditure", "life_expectancy"),
    ("vaccination_dtp3",   "infant_mortality"),
    ("water_access",       "maternal_mortality"),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (x_col, y_col) in zip(axes, pairs):
    pass  # TODO: scatter + Pearson r annotation
plt.tight_layout()
plt.show()

### Lab 3: Group Comparison
Create a `high_gdp` binary column: 1 if `gdp_per_capita` ≥ median, 0 otherwise.  
Compare mean, median, and std dev of `life_expectancy` between the two groups.  
Visualise the comparison with side-by-side boxplots.

In [ ]:
# Lab 3
df["high_gdp"] = (df["gdp_per_capita"] >= df["gdp_per_capita"].median()).astype(int)
# TODO: groupby high_gdp → mean, median, std of life_expectancy
# TODO: side-by-side boxplot

---
## 8. Assignment — Week 4

**Problem 1 (25 pts):** Write a function `outlier_summary(df, cols)` that returns a DataFrame with columns: `variable`, `Q1`, `Q3`, `IQR`, `lower_fence`, `upper_fence`, `n_outliers`, `outlier_pct`.  
Run it on all numeric columns of the Africa Health dataset.

**Problem 2 (25 pts):** Recreate the correlation heatmap but use Spearman correlation *and* annotate only cells where |ρ| > 0.4.

**Problem 3 (30 pts):** Choose **two** of the following analyses and produce a full bivariate plot (labelled axes, title, annotation):
- a) `maternal_mortality` vs `water_access` (scatter, by region)
- b) `hiv_prevalence` distribution by region (violin plot)
- c) `health_expenditure` vs `gdp_per_capita` (scatter, log x-axis)

**Problem 4 (20 pts):** Write a structured EDA paragraph (200–250 words) covering the main distributional patterns, top correlations, and at least one regional insight from the dataset.

In [ ]:
# Problem 1
def outlier_summary(df, cols):
    # TODO
    pass

result = outlier_summary(df, numeric_cols)
# print(result)

In [ ]:
# Problem 2 — Spearman heatmap, annotate |ρ| > 0.4
pass  # TODO

In [ ]:
# Problem 3 — Two bivariate analyses
pass  # TODO

**Problem 4 — Written response (200–250 words):**

> *Type your answer here.*

---
## Summary — Week 4

| Concept | Key Point |
|---|---|
| EDA purpose | Discover patterns before modelling |
| Descriptive stats | Mean/median for centre; IQR/std for spread; skewness/kurtosis for shape |
| Univariate tools | Histogram, KDE, box plot, bar chart |
| Bivariate tools | Scatter, correlation, violin plot |
| Multivariate tools | Pair plot, bubble chart, FacetGrid |
| Pearson vs Spearman | Linear vs monotonic; Spearman robust to outliers |
| EDA narrative | Overview → Quality → Univariate → Bivariate → Hypotheses |

**Next:** Week 5 — Data Visualisation Principles & Advanced Charts